# Graph Database Connection

In the following, we will show you how to connect and query data on Neo4j, using python. 

**IMPORTANT NOTE**

This notebook requires that you have access to a working version of Neo4j. In order to install Neo4j locally, we advise you to refer to the Neo4j webpage (https://neo4j.com/download/) or to use docker (https://hub.docker.com/_/neo4j).

In [14]:
with open("./dataset/movieCreationQuery.txt", "rb") as fid:
    lines = fid.readlines()

In [15]:
query = " ".join([line.decode("utf-8").replace("\n", "") for line in lines])

In [16]:
from neo4j import GraphDatabase

In [17]:
uri = "neo4j://localhost:7687"
driver = GraphDatabase.driver(uri, auth=("neo4j", "neo5j"))

In [18]:
def run_query(tx, query):
    return list(tx.run(query))

In [19]:
with driver.session() as session:
    session.write_transaction(run_query, query)

/tmp/ipykernel_1283239/469611421.py:2: DeprecationWarning: write_transaction has been renamed to execute_write
  session.write_transaction(run_query, query)


Query

In [20]:
query = "MATCH (n) RETURN count(*)"

In [21]:
with driver.session() as session:
    result = session.read_transaction(run_query, query)
[r for r in result]

/tmp/ipykernel_1283239/3167206639.py:2: DeprecationWarning: read_transaction has been renamed to execute_read
  result = session.read_transaction(run_query, query)


[<Record count(*)=171>]

### Using `graphdatascience`

In [10]:
import graphdatascience

/home/deusebio/.pyenv/versions/graph-machine-learning-310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
from graphdatascience import GraphDataScience

import os
host = os.environ.get("NEO4J_HOST", "localhost")

uri = f"bolt://{host}:7687"
gds = GraphDataScience(uri, auth=("neo4j", "neo5j"))

In [22]:
gds.run_cypher("MATCH (n) RETURN count(*);") 

,count(*)
0,171


### Using the analytics capabilities of `graphdatascience`

In [23]:
G = gds.graph.load_cora()

In [ ]:
gds.graph.list()

In [ ]:
G=gds.graph.get("cora")

In [ ]:
G.node_properties()

In [ ]:
gds.graph.nodeProperty.stream(gds.graph.get("cora"), node_property="subject")

In [ ]:
pr_result = gds.pageRank.mutate(G, mutateProperty="pagerank")

In [ ]:
print(f"Compute millis: {pr_result['computeMillis']}")
print(f"Node properties written: {pr_result['nodePropertiesWritten']}")
print(f"Centrality distribution: {pr_result['centralityDistribution']}")

In [9]:
G.node_properties()

In [ ]:
gds.graph.nodeProperties.stream(G, ["pagerank"], separate_property_columns=True)
# gds.graph.nodeProperties.write(G, ["pagerank"])

### Delete datasets

In [13]:
gds.graph.drop("cora")

graphName                                                             cora
database                                                             neo4j
databaseLocation                                                     local
memoryUsage                                                               
sizeInBytes                                                             -1
nodeCount                                                             2708
relationshipCount                                                     5429
configuration            {'readConcurrency': 4, 'undirectedRelationship...
density                                                           0.000741
creationTime                           2025-02-23T16:03:23.055419494+00:00
modificationTime                       2025-02-23T16:03:23.055419494+00:00
schema                   {'graphProperties': {}, 'nodes': {'Paper': {'s...
schemaWithOrientation    {'graphProperties': {}, 'nodes': {'Paper': {'s...
Name: 0, dtype: object

In [9]:
with driver.session() as session:
    result = session.write_transaction(run_query, "MATCH (n)-[e]-() DELETE n, e")

/tmp/ipykernel_1283239/1407641033.py:2: DeprecationWarning: write_transaction has been renamed to execute_write
  result = session.write_transaction(run_query, "MATCH (n)-[e]-() DELETE n, e")
